# Lab 03-2 - Principal Component Analysis

This is a short lab that will cover the tools available in `sklearn` for principal components analysis with a focus on dimension reduction.

In [ ]:
## Standard Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn

Examples throughout the lab will use a large-scale personality data set obtained from the Open-Source Psychometrics Project: https://openpsychometrics.org/. These data contain several thousand responses to an online personality survey consisting of 50 statements rated on a 1 to 5 point [likert scale](https://en.wikipedia.org/wiki/Likert_scale) designed to measure the [Big 5 Personality Traits](https://en.wikipedia.org/wiki/Big_Five_personality_traits):

![](https://upload.wikimedia.org/wikipedia/commons/thumb/c/c0/Wiki-grafik_peats-de_big_five_ENG.svg/500px-Wiki-grafik_peats-de_big_five_ENG.svg.png)

In [ ]:
## Big 5 data
bf = pd.read_csv("../data/big5data.csv", sep='\t')

## Split the personality questions from the demographics
bf_q = bf.drop(['race','age','engnat','gender','hand','source','country'], axis=1)
bf_demo = bf[['race','age','engnat','gender','hand','source','country']]

## Note this data has a large n relative to p (the responses to the 50 questions)
bf_q.shape

In [ ]:
# Here are the first 10 rows of the demographic data:
bf_demo.head(10)

In [ ]:
## Here are the first ten rows of the responses. Notice how the survey responses are coded in a Likert scale where: 5 = "Agree", 3 = "Neutral",  1 = "Disagree".
bf_q.head(10)

What do these columns mean? Click the data's codebook [here](../data/big5_code.txt) to see what these 50 questions actually are. 

## Part 1 - Dimension Reduction

We'd expect a high degree of covariance/correlation in how participants tend to rate many of these statements. Thus, we might seek a lower dimensional representation of these data as we can retain most of information about the personalities of respondents using fewer than 50 variables. i.e. can we reduce $p=50$ to something smaller but still capture most of the information in the dataset. 

In `sklearn` the primary function used to perform principal component analysis is `PCA()`:

In [ ]:
from sklearn.decomposition import PCA
pca_bfq = PCA().fit(bf_q)

Similar to the clustering functions we've worked with, we must use the `.fit()` method to perform PCA on our data. You should also note that we did not standardize these data prior to fitting because they are already on a standardized 1-5 scale. 

The `explained_variance_ratio_` attribute of our fitted PCA object stores the variance explained by each component:

In [ ]:
## Variance explained by components 1-50 out of 50
var_explained = pca_bfq.explained_variance_ratio_[0:50]
var_explained

**Question 1**: 

- **Part A**: Create a *scree plot* showing the variance explained by each component. Use this plot to identify the number of components $k$ that you think should be retained for these data.
- **Part B**: Create a plot showing the *cummulative* variance explained by each component. Ex: for $x$ = 5, y = the cummulative variance explained by the first 5 principal components. Hint: your plot should plateau at around $y$ = 1.  
- **Part C**: Looking at the plot from Part B, what is the total proportion of the variance explained by the first $k$ principal components you identified in Part A. 


## Part 2 - Scores

This questionnaire was designed to measure the [Big 5 Personality Traits](https://en.wikipedia.org/wiki/Big_Five_personality_traits), so we might opt to retain only 5 principal components.

To achieve this we need to refit the PCA with the argument `n_components=5`, as the `fit_transform()` method of PCA depends upon the number of components specified when the object is created:

In [ ]:
## Perform the dimension reduction
pca_bfq_5comp = PCA(n_components=5).fit_transform(bf_q)

## Verify results
pca_bfq_5comp.shape

Notice that while the original data set was 19719 x 50, the new *dimension reduced* dataset is 19719 x 5. 

The `fit_transform()` method is used to fit a PCA model with a specified number of principal components and return the lower dimensional representation of the input data utilizing those components.  You should note that `pca_bfq_5comp` contains the "scores" (ie: coordinates) of each data-point in the retained principal component dimensions.

**Question 2**: Create a scatterplot displaying scores for the first and second principal components. Color each data-point by the demographic variable `engnat` (English-speaking nationality) and briefly comment upon whether you can visually see any noticeable  differences in these scores among English-speaking nationalities compared to non-English-speaking nationalities.

## Part 3 - Loadings 

In this application we're interested in assigning meaningful labels to these 5 dimensions that were derived using PCA. This can be done by inspecting the most influential loadings in each component.

In [ ]:
## Loadings for a particular principal component (sorted by abs magnitude), here the first principal component PC1
PC1_loadings = pd.DataFrame({'Question': bf_q.columns, 'PC1': pca_bfq.components_[0]})
print(PC1_loadings.sort_values(by='PC1', ascending=False, key=abs).head(10))

Referring to the data's codebook [here](../data/big5_code.txt), we can see that the 4 most influential statements in determining where an individual falls in the first principal component dimension are:
    
1. E7: I talk to a lot of different people at parties.
2. E3: I feel comfortable around people.
3. E5: I start conversations.
4. E10: I am quiet around strangers.

Additionally, you might notice that 7 of 10 top contributors are statements with the "E" label. This is intentional, the creators of this questionnaire designed these statements to measure extroversion, which emerges as the most prominent personality dimension in these data.

**Question 3**:

Using the most influential loadings, label each of the first five principal component dimension in `pca_bfq` as one of the Big Five personality traits: E = extroversion, O = openness, A = agreeableness, C = conscientiousness, and N = neuroticism.

## Part 4 - On your own

The code given below loads responses to an online questionnaire  aimed at capturing "dark triad" personality traits, which are:

1. machiavellianism (a manipulative attitude)
1. narcissism (excessive self-love)
1. psychopathy (lack of empathy)

For more information you can visit this link: http://openpsychometrics.org/tests/SD3/

Like the Big 5 dataset, the statements are labeled according to which trait they were intended to measure.

**Question 4**:

- **Part A**: Perform PCA on the survey items in the dark triad data and plot the variance explained by each component. Based upon this plot, does it seem reasonable to conclude that 3 underlying factors are being measured by these questions?
- **Part B**: Using the principal component loadings, assign a label (ie: M = machiavellianism, N = narcissism, P = psychopathy) to each of the first three principal components.
- **Part C (Bonus)**: For any two countries of your choosing, create a data visualization showing the distributions of your choice of dark triad traits across the respondents from each country.

For your reference, the image below (from the appendix of Jones and Paulhus (2014) https://journals.sagepub.com/doi/full/10.1177/1073191113514105) displays the individual survey items.

In [ ]:
## Code to load the data
dt = pd.read_csv("../data/dark_triad.csv", sep='\t')

In [ ]:
## Screenshot of the Dark Triad survey items 
from IPython.display import Image
Image("../images/dark.PNG")